# Fast Prompting en Acción: automatización de contenido para e-commerce

**Autora:** Amparo Sanz  
**Diplomatura en Data Science – Prompt Engineering**  
**Comisión:** #96165

## POC
Esta notebook continúa la Preentrega 1 y demuestra cómo diferentes configuraciones de prompts pueden utilizarse para generar contenido promocional para un e-commerce.

La demostración se concentra en **texto → texto** para poder comparar prompts, medir llamadas a la API y mantener una implementación simple y reproducible.


## 1. Problema y objetivo

Los pequeños e-commerce necesitan publicar contenido con frecuencia, pero pueden no contar con tiempo, conocimientos de marketing o presupuesto para delegar la tarea.

**Objetivo de la POC:** recibir información estructurada de un producto y generar una publicación comercial adaptada al público, canal, tono y objetivo.

Se compararán:
1. **Prompt básico:** instrucción breve con poco contexto.
2. **Fast Prompt estructurado:** rol + contexto + tarea + restricciones + formato.

> La comparación requiere dos consultas únicamente durante el experimento. El flujo final utiliza una sola.


## 2. Configuración

La API key se obtiene desde una variable de entorno para evitar incluir credenciales en el código o en GitHub.

La notebook puede recorrerse con `EJECUTAR_API = False` sin realizar ninguna consulta.


In [ ]:
import os
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")
cliente = OpenAI(api_key=api_key)

usar_api = False

print("API configurada:", api_key is not None)
print("Ejecutar API:", usar_api)

## 3. Datos de entrada

Las variables se encuentran separadas del prompt. Esto hace que la solución sea reutilizable sin reescribir las instrucciones.


In [ ]:
producto = {
    "nombre": "Zapatillas Urban Street",
    "caracteristicas": "zapatillas urbanas cómodas, diseño moderno y suela liviana",
    "publico": "jóvenes de 18 a 30 años interesados en moda urbana",
    "canal": "Instagram",
    "tono": "cercano, moderno y dinámico",
    "objetivo": "generar interés y dirigir potenciales clientes a la tienda online"
}

for clave, valor in producto.items():
    print(f"{clave.capitalize()}: {valor}")


## 4. Prompt básico

Esta versión contiene el objetivo principal, pero deja varias decisiones abiertas al modelo. Sirve como línea de base para la comparación.


In [ ]:
def crear_prompt_basico(datos):
    return (
        f"Crea una publicación para {datos['canal']} promocionando "
        f"{datos['nombre']}. Características: {datos['caracteristicas']}."
    )

prompt_basico = crear_prompt_basico(producto)
print(prompt_basico)


## 5. Fast Prompt estructurado

La segunda versión utiliza:

- **Rol:** especialista en marketing digital y e-commerce.
- **Contexto:** datos concretos del producto.
- **Objetivo:** acción que debe favorecer la pieza.
- **Restricciones:** extensión, cantidad de hashtags y prohibición de inventar información.
- **Formato:** estructura fija de salida.

Esto busca reducir ambigüedad y aumentar consistencia.


In [ ]:
def crear_fast_prompt(datos):
    return f"""
ROL
Actúa como especialista en marketing digital para pequeños e-commerce.

CONTEXTO
Producto: {datos['nombre']}
Características: {datos['caracteristicas']}
Público objetivo: {datos['publico']}
Canal: {datos['canal']}
Tono: {datos['tono']}
Objetivo: {datos['objetivo']}

TAREA
Redacta una publicación promocional lista para utilizar en {datos['canal']}.

RESTRICCIONES
- Escribe entre 70 y 110 palabras.
- Mantén el tono indicado.
- Destaca únicamente características proporcionadas en el contexto.
- No inventes descuentos, precios, stock, envíos ni beneficios no informados.
- Incluye un llamado a la acción breve.
- Incluye exactamente 5 hashtags relevantes.
- Evita explicaciones sobre cómo construiste la respuesta.

FORMATO DE SALIDA
PUBLICACIÓN:
<texto>

CTA:
<una frase>

HASHTAGS:
<5 hashtags>
""".strip()

prompt_fast = crear_fast_prompt(producto)
print(prompt_fast)


## 6. Función de consulta

Todas las consultas pasan por una única función. Esto simplifica el código y permite registrar el uso informado por la API.

La API de OpenAI es un servicio con costo según el modelo y uso. Por eso la ejecución queda desactivada por defecto.


In [ ]:
def consultar_modelo(prompt):
    if not usar_api:
        return {
            "texto": "[SIMULACIÓN] La API está desactivada.",
            "input_tokens": None,
            "output_tokens": None,
            "total_tokens": None
        }

    response = cliente.responses.create(
        model="gpt-4o",
        input=prompt
    )

    usage = getattr(response, "usage", None)

    return {
        "texto": response.output_text,
        "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
        "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
        "total_tokens": getattr(usage, "total_tokens", None) if usage else None
    }


## 7. Experimento: comparación de prompts

Para demostrar la optimización se ejecutan ambos prompts sobre **el mismo producto**.

**Cantidad de consultas del experimento: 2.**

Esta sección no representa el flujo productivo; existe para evaluar el efecto de la estrategia de prompting.


In [ ]:
if usar_api:
    resultado_basico = consultar_modelo(prompt_basico)
    resultado_fast = consultar_modelo(prompt_fast)

    print("=== RESULTADO PROMPT BÁSICO ===")
    print(resultado_basico["texto"])
    print("\nTokens:", resultado_basico["total_tokens"])

    print("\n=== RESULTADO FAST PROMPT ===")
    print(resultado_fast["texto"])
    print("\nTokens:", resultado_fast["total_tokens"])
else:
    print("Experimento preparado. No se realizaron consultas a la API.")
    print("Para ejecutarlo, cambiar usar_api = True y volver a correr las celdas.")


**Resultados de demostración:** debido a que la ejecución mediante API requiere créditos, se incluyen a continuación ejemplos de salida para visualizar la comparación entre ambos enfoques. El código de integración con la API queda implementado en la notebook.

In [ ]:
resultado_basico_ejemplo = """
¡Renová tu estilo con Zapatillas Urban Street!

Descubrí unas zapatillas urbanas cómodas, con diseño moderno
y suela liviana. Ideales para acompañarte todos los días
y completar tu look urbano.

Conocé Zapatillas Urban Street y sumá estilo a tus pasos.
"""

resultado_fast_ejemplo = """
PUBLICACIÓN:
Dale un toque urbano a tu día con Zapatillas Urban Street.
Su diseño moderno, comodidad y suela liviana las convierten
en una opción ideal para quienes buscan acompañar su estilo
con un calzado pensado para el ritmo cotidiano.

CTA:
Descubrí Zapatillas Urban Street en nuestra tienda online.

HASHTAGS:
#ZapatillasUrbanas #ModaUrbana #EstiloUrbano #StreetStyle #Zapatillas
"""

print("=== RESULTADO PROMPT BÁSICO ===")
print(resultado_basico_ejemplo)

print("=== RESULTADO FAST PROMPT ===")
print(resultado_fast_ejemplo)

## 8. Evaluación

La calidad no se evalúa solamente por si el texto “suena bien”. Se utilizan criterios relacionados con el problema planteado.

| Criterio | Qué se observa |
|---|---|
| Adecuación al canal | Si el contenido funciona como publicación de Instagram |
| Público objetivo | Si el lenguaje está adaptado a jóvenes de 18 a 30 años |
| Tono | Si respeta un estilo cercano, moderno y dinámico |
| CTA | Si contiene una acción clara |
| Hashtags | Si cumple exactamente con los 5 solicitados |
| Fidelidad | Si evita inventar precios, descuentos u otros datos |
| Formato | Si devuelve las secciones solicitadas |

### Hipótesis
Se espera que el prompt estructurado obtenga mayor cumplimiento porque reduce decisiones implícitas y comunica explícitamente contexto, restricciones y formato.


**Comparacion de Resultados**

| Criterio              | Prompt básico | Fast Prompt |
| --------------------- | ------------- | ----------- |
| Adecuación al canal   | Parcial       | Cumple      |
| Público objetivo      | Poco definido | Cumple      |
| Tono solicitado       | Parcial       | Cumple      |
| CTA                   | Parcial       | Cumple      |
| 5 hashtags            | No cumple     | Cumple      |
| Fidelidad a los datos | Cumple        | Cumple      |
| Formato solicitado    | No cumple     | Cumple      |

**Análisis**: El Fast Prompt obtuvo un mayor cumplimiento de los criterios definidos. Al incorporar contexto, público objetivo, tono, restricciones y un formato de salida específico, la respuesta resulta más controlada y consistente. El prompt básico genera un contenido utilizable, pero deja más decisiones a criterio del modelo y no garantiza elementos como la cantidad de hashtags o la estructura de la respuesta.



In [ ]:
criterios = {
    "Adecuación al canal": "¿El contenido es apropiado para Instagram?",
    "Público objetivo": "¿El lenguaje se adapta al público definido?",
    "Tono": "¿Respeta el tono solicitado?",
    "CTA": "¿Incluye un llamado a la acción claro?",
    "Hashtags": "¿Incluye exactamente cinco hashtags?",
    "Fidelidad": "¿Evita inventar información comercial?",
    "Formato": "¿Respeta la estructura solicitada?"
}

print("Checklist de evaluación:")
for i, (criterio, pregunta) in enumerate(criterios.items(), 1):
    print(f"{i}. {criterio}: {pregunta}")


## 9. Flujo optimizado para producción

Una vez finalizado el experimento, no es necesario seguir generando una versión básica.

El flujo real es:

**datos → Fast Prompt → 1 consulta → contenido**

Por lo tanto, cada pieza de contenido necesita solamente **una llamada a la API**.


In [ ]:
def generar_contenido(datos):
    prompt = crear_fast_prompt(datos)
    return consultar_modelo(prompt)

if usar_api:
    contenido_final = generar_contenido(producto)
    print(contenido_final["texto"])
    print("\nUso total:", contenido_final["total_tokens"], "tokens")
else:
    print("Flujo productivo preparado: 1 consulta por contenido.")


## 10. Prueba con otro producto

Una ventaja del template es que puede reutilizarse cambiando solamente los datos.


In [ ]:
otro_producto = {
    "nombre": "Mochila CityPack",
    "caracteristicas": "mochila urbana compacta con compartimento acolchado para notebook",
    "publico": "estudiantes universitarios y jóvenes profesionales",
    "canal": "Instagram",
    "tono": "práctico y cercano",
    "objetivo": "presentar el producto y generar visitas a la tienda online"
}

print(crear_fast_prompt(otro_producto))


## 11. Análisis de costos y eficiencia

La notebook evita consultas innecesarias:

- **Construir o visualizar prompts:** 0 consultas.
- **Experimento comparativo:** 2 consultas.
- **Generación final:** 1 consulta por contenido.
- **Cambio de producto:** no requiere una consulta hasta que el usuario decida generar.

Además, la función registra tokens de entrada y salida cuando están disponibles. De esta forma, el costo puede analizarse con la tarifa vigente del modelo elegido sin dejar valores monetarios desactualizados dentro del proyecto.

### Decisión de diseño
No se solicita al modelo que genere varias alternativas para luego elegir una automáticamente. Para la POC, una respuesta bien condicionada es suficiente y evita multiplicar el consumo.


## 12. Comparación con la Preentrega 1

La Preentrega 1 proponía automatizar diferentes contenidos para redes sociales y tiendas online, además de explorar generación de imágenes.

En esta segunda entrega la propuesta se mejora de cuatro maneras:

1. **Se acota el problema:** la POC se concentra en una publicación comercial.
2. **Se implementa código:** la idea deja de ser únicamente conceptual.
3. **Se parametriza el prompt:** el mismo sistema funciona con diferentes productos.
4. **Se analiza eficiencia:** se diferencia el costo experimental del flujo productivo.

La generación de imágenes se conserva como posible ampliación futura, pero no es necesaria para demostrar Fast Prompting.


## 13. Conclusiones

La experimentación permite observar que un prompt estructurado ofrece mayor control sobre la respuesta que una instrucción genérica.

La mejora no consiste simplemente en agregar texto al prompt. La información se organiza de acuerdo con su función: rol, contexto, tarea, restricciones y formato. Esto facilita la reutilización y disminuye ambigüedades.

La POC también muestra que experimentar con dos prompts no implica duplicar permanentemente el costo de la solución. Las dos llamadas pertenecen a la etapa de evaluación; una implementación productiva utiliza solamente el prompt seleccionado.

### Posibles mejoras futuras
- formulario interactivo con `ipywidgets`;
- generación de diferentes tipos de contenido;
- evaluación automática con métricas;
- almacenamiento de resultados;
- generación opcional de imágenes mediante una herramienta compatible.
